# inbox

> Braincave inbox triage — read notes, apply Jelle's verdict, keep an undo stack.

Stage 1 of the human-gated inbox pipeline. Jelle makes the keep / delete / too-hard call
himself; this module only does what he decided. It never guesses a `note-status`, never
proposes a Hopsakee.Decimal, and never moves a file.

Two rules shape every function here:

- **The status vocabulary lives in the vault, not in this code.** `load_note_statuses`
  reads `Extra/metadata/notestatus-classes.md` so the picker can never drift from the
  eight values Jelle actually uses.
- **This pipeline writes exactly one key, and that key is hd.** No `hd-domain`, no
  `hd-proposed`, no `hoggle-status`. The value is refined in place: `d6` here, then
  `d6.75` once a category is picked.

In [ ]:
#| default_exp inbox

In [ ]:
#| export
import json
import re
import subprocess
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

## Paths and vocabulary

Everything authoritative is read from disk. `hd-structure.json` is rebuilt nightly at 02:00
by step 2.5 of `run-braincave-nightly.sh`, so it is always in lockstep with the vault's real
folder structure — which is exactly why this module reads it instead of the static
`config/d*.json` files `filesorter` uses for the Downloads tree.

In [ ]:
#| export
VAULT = Path.home() / "Obsidian" / "Braincave"
INBOX = VAULT / "0_Inbox"

#: Authoritative Hopsakee.Decimal structure. Rebuilt nightly; never hand-edited.
HD_STRUCTURE = Path.home() / ".claude" / "skills" / "_HD" / "hd-structure.json"

#: Canonical `note-status` vocabulary, owned by the vault.
NOTESTATUS_FILE = VAULT / "Extra" / "metadata" / "notestatus-classes.md"

LEDGER_DIR = Path.home() / ".claude" / "PAI" / "MEMORY" / "LEARNING"
LEDGER = LEDGER_DIR / "hd-choices.jsonl"
PENDING_DELETES = LEDGER_DIR / "inbox-pending-deletes.jsonl"
LEDGER_V = 1

#: Jelle's processing order (decided 2026-07-28). Small, high-signal folders first;
#: the 847-note NoteStation dump last. Folders not listed sort after these.
INBOX_ORDER = [
    "Phone", "Hoggle", "NewNotes", "Notes", "Readwise",
    "Snipd", "Atlas", "Sources", "Omnivore", "zz_NoteStationDumb",
]

#: `d0 Didymus` is a generated read-only mirror of `PAI/USER/` — no inbox note is ever
#: filed there, so it must not appear in the domain picker.
EXCLUDED_DOMAINS = {"d0"}

#: Frontmatter keys worth showing in the preview header. Everything else is noise
#: when the question is only "keep, delete, or too hard?".
PREVIEW_KEYS = ("hd", "note-status", "tags", "source", "created", "author", "url")

FM_DELIM = "---"
TODO_TASK = "- [ ] Organise this note [15m]"

## Hopsakee.Decimal structure

`hd-structure.json` uses `id` for the domain key (`d6`), **not** `domain_id` — so
`filesorter.domain.JDDomain` cannot parse it (it would derive `domain_id` from the first
word of `name` and produce `Programming`). Hence a dedicated loader.

In [ ]:
#| export
@dataclass
class HDArea:
    """A Hopsakee.Decimal area, e.g. `70-79 LLM en Generative AI`."""
    id: str
    name: str
    categories: list[dict] = field(default_factory=list)


@dataclass
class HDDomain:
    """A Hopsakee.Decimal domain, e.g. `d6 Programming en Development`."""
    id: str
    name: str
    areas: list[HDArea] = field(default_factory=list)

    @property
    def label(self) -> str: return f"{self.id} {self.name}"


def load_hd_domains(path: Path = HD_STRUCTURE, exclude: set[str] = EXCLUDED_DOMAINS) -> list[HDDomain]:
    """Load domains from the authoritative structure cache, dropping generated mirrors."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    domains = []
    for d in data.get("domains", []) or []:
        did = (d.get("id") or "").strip()
        if not did or did in exclude: continue
        areas = [
            HDArea(id=(a.get("id") or ""), name=(a.get("name") or ""), categories=(a.get("categories") or []))
            for a in (d.get("areas") or [])
        ]
        domains.append(HDDomain(id=did, name=(d.get("name") or ""), areas=areas))
    return sorted(domains, key=lambda d: d.id)


def hd_structure_age_days(path: Path = HD_STRUCTURE) -> float:
    """Days since the structure cache was regenerated.

    `_BRAINCAVE_INBOX_SORT` aborts above 14 days because a stale tree can name categories
    that no longer exist. Triage only writes a *domain*, so it is far more tolerant — but
    the caller should still warn.
    """
    return (datetime.now().timestamp() - Path(path).stat().st_mtime) / 86400

In [ ]:
#| export
def load_note_statuses(path: Path = NOTESTATUS_FILE) -> list[str]:
    """Read the canonical `note-status` values from the vault, in file order.

    Deliberately not hardcoded: Jelle owns this list, and a copy here would silently
    drift the moment he adds a status.
    """
    out = []
    for raw in Path(path).read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith(("#", ">", "-", "*", "|")): continue
        out.append(line)
    return out

## Reading a note

The parser is line-based and keeps the original lines verbatim. That is a hard requirement:
these are 2657 hand-written notes, and a PyYAML round-trip would reorder keys, restyle
values and rewrite quoting across the whole inbox.

In [ ]:
#| export
@dataclass
class Note:
    """A parsed note that remembers its exact original lines."""
    path: Path
    lines: list[str]
    fm_start: int | None = None
    fm_end: int | None = None

    @property
    def has_frontmatter(self) -> bool:
        return self.fm_start is not None and self.fm_end is not None

    @property
    def fm_lines(self) -> list[str]:
        return self.lines[self.fm_start + 1:self.fm_end] if self.has_frontmatter else []

    @property
    def body_lines(self) -> list[str]:
        return self.lines[self.fm_end + 1:] if self.has_frontmatter else self.lines

    def get(self, key: str, default: str | None = None) -> str | None:
        """Value of a top-level frontmatter key, or `default` when absent/empty."""
        pat = re.compile(rf"^{re.escape(key)}:\s*(.*)$")
        for line in self.fm_lines:
            m = pat.match(line)
            if m: return m.group(1).strip() or default
        return default


def parse_note(path: Path) -> Note:
    """Parse a markdown note, locating its frontmatter fence if it has one."""
    p = Path(path)
    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()
    fm_start = fm_end = None
    if lines and lines[0].strip() == FM_DELIM:
        for i, line in enumerate(lines[1:], start=1):
            if line.strip() == FM_DELIM:
                fm_start, fm_end = 0, i
                break
    return Note(path=p, lines=lines, fm_start=fm_start, fm_end=fm_end)

In [ ]:
#| export
def note_preview(note: Note, n: int = 20) -> tuple[list[str], list[str]]:
    """Return `(header, body)` — interesting frontmatter keys, then `n` non-empty body lines.

    Splitting these matters: a raw first-20-lines preview of a vault note shows only YAML,
    which tells you nothing about what the note is.
    """
    header = [f"{k}: {v}" for k in PREVIEW_KEYS if (v := note.get(k))]
    body: list[str] = []
    for line in note.body_lines:
        s = line.strip()
        if s and s != FM_DELIM:
            body.append(s)
            if len(body) >= n: break
    return header, body

## Backlinks

The single most decision-relevant signal, and the one neither `filesorter` nor
`_BRAINCAVE_INBOX_SORT` surfaces: **a note with inbound links is connected, so deleting it
breaks something.** Counting is one ripgrep pass over the vault at startup rather than a
grep per note, which would be 2657 subprocesses.

In [ ]:
#| export
WIKILINK = re.compile(r"\[\[([^\]|#]+)")


def build_backlink_index(vault: Path = VAULT, timeout: int = 120) -> dict[str, int]:
    """Count inbound wikilink targets across the whole vault in a single pass.

    Keyed on the link target's final path segment, which is how Obsidian resolves
    `[[Some Note]]`. Falls back to pure Python when ripgrep is unavailable.
    """
    counts: dict[str, int] = {}

    def tally(text: str) -> None:
        for target in WIKILINK.findall(text):
            key = target.split("/")[-1].strip()
            if key: counts[key] = counts.get(key, 0) + 1

    try:
        res = subprocess.run(
            ["rg", "-oI", "--no-filename", r"\[\[[^\]]+\]\]", "-g", "*.md", str(vault)],
            capture_output=True, text=True, timeout=timeout,
        )
        # rg exits 1 on "no matches", which is not an error here.
        if res.returncode in (0, 1):
            tally(res.stdout)
            return counts
    except Exception:
        pass

    for p in Path(vault).rglob("*.md"):
        try: tally(p.read_text(encoding="utf-8", errors="replace"))
        except Exception: continue
    return counts


def backlink_count(index: dict[str, int], path: Path) -> int:
    """Inbound link count for `path`, from a prebuilt index."""
    return index.get(Path(path).stem, 0)

In [ ]:
#| export
def list_candidates(inbox: Path = INBOX, order: list[str] = INBOX_ORDER) -> list[Path]:
    """Every inbox note, in Jelle's processing order then alphabetically within a folder."""
    inbox = Path(inbox)
    rank = {name: i for i, name in enumerate(order)}
    notes = [p for p in inbox.rglob("*.md") if not p.name.startswith(".")]

    def sort_key(p: Path):
        parts = p.relative_to(inbox).parts
        top = parts[0] if len(parts) > 1 else ""
        return (rank.get(top, len(order)), str(p).lower())

    return sorted(notes, key=sort_key)

## Writing a verdict

`set_frontmatter` is the only writer, and it edits in place at an existing path — which is
the one vault mutation allowed outside the `obsidian` CLI, because no wikilink can break
when neither the filename nor the folder changes. Anything that *moves* a note stays the
CLI's job.

In [ ]:
#| export
def set_frontmatter(path: Path, updates: dict[str, str]) -> None:
    """Set frontmatter keys in place, preserving every other line verbatim.

    Existing keys are replaced where they sit; new keys are appended just before the closing
    fence; a note with no frontmatter gets a fresh block. Never reorders, never reformats.
    """
    p = Path(path)
    note = parse_note(p)
    original = p.read_text(encoding="utf-8", errors="replace")

    if not note.has_frontmatter:
        block = [FM_DELIM, *[f"{k}: {v}" for k, v in updates.items()], FM_DELIM]
        lines = block + note.lines
    else:
        lines = list(note.lines)
        fm_end = note.fm_end
        for key, value in updates.items():
            pat = re.compile(rf"^{re.escape(key)}:\s*(.*)$")
            hit = next((i for i in range(note.fm_start + 1, fm_end) if pat.match(lines[i])), None)
            if hit is not None:
                lines[hit] = f"{key}: {value}"
            else:
                lines.insert(fm_end, f"{key}: {value}")
                fm_end += 1

    trailing = "\n" if original.endswith("\n") or not original else ""
    p.write_text("\n".join(lines) + trailing, encoding="utf-8")


def append_task_line(path: Path, when: date | None = None, text: str = TODO_TASK) -> str:
    """Append a `_DAILY_PLAN`-schedulable task line to the note itself.

    `_DAILY_PLAN` schedules work by putting `⏳ YYYY-MM-DD` on a task line inside its source
    note, so a "this one is too hard right now" verdict becomes a task that lives in the note
    it is about — no separate task file, no wikilink plumbing. Defaults to tomorrow.
    """
    when = when or (date.today() + timedelta(days=1))
    line = f"{text} ⏳ {when.isoformat()}"
    p = Path(path)
    current = p.read_text(encoding="utf-8", errors="replace")
    sep = "" if current.endswith("\n") or not current else "\n"
    p.write_text(f"{current}{sep}{line}\n", encoding="utf-8")
    return line

## The ledger

Every verdict is logged. The pick-vs-proposal comparison this enables is the point of the
whole pipeline: a category Jelle keeps overriding is a category whose *description* is wrong,
and that description lives in the vault MOC where the next nightly rebuild will pick it up.

Field order (`timestamp`, `v`, `tool`, `event_type`) follows the PAI observability rule so
cross-file `jq` consumers work without special-casing this file.

In [ ]:
#| export
def _append_jsonl(path: Path, record: dict) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def log_event(event_type: str, note: Path, **payload) -> dict:
    """Append one triage event to the ledger and return the record."""
    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "v": LEDGER_V,
        "tool": "inbox-triage",
        "event_type": event_type,
        "note": str(Path(note)),
    }
    record.update(payload)
    _append_jsonl(LEDGER, record)
    return record


def queue_delete(path: Path) -> dict:
    """Record an intent to delete, without deleting.

    Deleting is the one verdict the `obsidian` CLI must own — it routes to the OS trash and
    keeps the vault's link graph consistent — and that CLI needs the desktop app running,
    which triage deliberately does not. Queuing also means a session of 200 `d` presses stays
    reviewable before anything is actually gone.
    """
    record = log_event("delete_queued", path)
    _append_jsonl(PENDING_DELETES, {**record, "event_type": "delete_pending"})
    return record


def _unqueue_delete(path: Path) -> None:
    """Drop `path` from the pending-delete queue (used by undo)."""
    p = Path(PENDING_DELETES)
    if not p.exists(): return
    target = str(Path(path))
    kept = []
    for line in p.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        try: keep = json.loads(line).get("note") != target
        except json.JSONDecodeError: keep = True
        if keep: kept.append(line)
    p.write_text("\n".join(kept) + ("\n" if kept else ""), encoding="utf-8")


def pending_deletes() -> list[Path]:
    """Notes currently queued for deletion."""
    p = Path(PENDING_DELETES)
    if not p.exists(): return []
    out = []
    for line in p.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        try: out.append(Path(json.loads(line)["note"]))
        except (json.JSONDecodeError, KeyError): continue
    return out

## Applying verdicts

`Triage` snapshots a note's bytes before mutating it, so undo is an exact restore rather than
a best-effort inverse edit.

In [ ]:
#| export
@dataclass
class Action:
    """One applied verdict, with enough state to undo it exactly."""
    note: Path
    kind: str
    before: str | None = None
    delete_queued: bool = False


class Triage:
    """Applies Jelle's verdicts and keeps a per-session undo stack."""

    def __init__(self):
        self.stack: list[Action] = []
        self.counts: dict[str, int] = {}

    def _snapshot(self, path: Path) -> str:
        return Path(path).read_text(encoding="utf-8", errors="replace")

    def _push(self, action: Action) -> None:
        self.stack.append(action)
        self.counts[action.kind] = self.counts.get(action.kind, 0) + 1

    def keep(self, path: Path, status: str, hd: str) -> str:
        """Record a keeper: Jelle's chosen `note-status` plus the domain he picked."""
        before = self._snapshot(path)
        set_frontmatter(path, {"note-status": status, "hd": hd})
        log_event("keep", path, note_status=status, hd=hd)
        self._push(Action(Path(path), "keep", before))
        return f"{status} · hd: {hd}"

    def delete(self, path: Path) -> str:
        queue_delete(path)
        self._push(Action(Path(path), "delete", None, delete_queued=True))
        return "queued for delete"

    def todo(self, path: Path, when: date | None = None) -> str:
        before = self._snapshot(path)
        line = append_task_line(path, when)
        log_event("todo", path, task=line)
        self._push(Action(Path(path), "todo", before))
        return line

    def skip(self, path: Path) -> str:
        log_event("skip", path)
        self._push(Action(Path(path), "skip", None))
        return "skipped"

    def can_undo(self) -> bool:
        return bool(self.stack)

    def undo(self) -> Action | None:
        """Restore the last note byte-for-byte and unqueue any delete it had."""
        if not self.stack: return None
        action = self.stack.pop()
        if action.before is not None:
            action.note.write_text(action.before, encoding="utf-8")
        if action.delete_queued:
            _unqueue_delete(action.note)
        log_event("undo", action.note, undone=action.kind)
        self.counts[action.kind] = max(0, self.counts.get(action.kind, 1) - 1)
        return action

    def summary(self) -> str:
        if not self.counts: return "nothing triaged"
        return " | ".join(f"{n} {kind}" for kind, n in sorted(self.counts.items()))

## The `obsidian` CLI edges

Two operations must go through the CLI: opening a note (needs the running app anyway) and
draining the delete queue. `drain_deletes` defaults to a dry run — deleting 200 notes should
never be a side effect of a typo.

In [ ]:
#| export
def obsidian_relpath(path: Path, vault: Path = VAULT) -> str:
    """Vault-relative path, which is what `obsidian path=` expects."""
    return str(Path(path).resolve().relative_to(Path(vault).resolve()))


def obsidian_open(path: Path, vault_name: str = "Braincave") -> subprocess.CompletedProcess:
    """Open a note in the Obsidian desktop app."""
    return subprocess.run(
        ["obsidian", "open", f"path={obsidian_relpath(path)}", f"vault={vault_name}"],
        capture_output=True, text=True, timeout=30,
    )


def drain_deletes(dry_run: bool = True, vault_name: str = "Braincave") -> list[tuple[Path, str]]:
    """Apply queued deletes via `obsidian delete` (OS trash — recoverable).

    Never passes `permanent`. Requires the Obsidian desktop app to be running. Returns
    `(note, outcome)` per queued note; on a real run the queue is rewritten to hold only
    the failures, so re-running retries exactly those.
    """
    results: list[tuple[Path, str]] = []
    failed: list[Path] = []
    for note in pending_deletes():
        if dry_run:
            results.append((note, "would delete"))
            continue
        try:
            res = subprocess.run(
                ["obsidian", "delete", f"path={obsidian_relpath(note)}", f"vault={vault_name}"],
                capture_output=True, text=True, timeout=30,
            )
            ok = res.returncode == 0
            results.append((note, "deleted" if ok else f"failed: {(res.stderr or res.stdout).strip()[:80]}"))
            if ok: log_event("delete_applied", note)
            else: failed.append(note)
        except Exception as e:
            results.append((note, f"failed: {e}"))
            failed.append(note)

    if not dry_run:
        p = Path(PENDING_DELETES)
        p.write_text("", encoding="utf-8")
        for note in failed: queue_delete(note)
    return results

## Smoke tests

These run against the real vault, read-only, so they double as a health check on the
contracts this module depends on.

In [ ]:
statuses = load_note_statuses()
print(statuses)
assert len(statuses) >= 5, "note-status vocabulary looks empty"
assert any("vervallen" in s for s in statuses)

In [ ]:
domains = load_hd_domains()
print([d.label for d in domains], f"| structure is {hd_structure_age_days():.1f} days old")
assert domains, "no domains loaded"
assert all(d.id != "d0" for d in domains), "d0 must never be an inbox destination"

In [ ]:
cands = list_candidates()
print(f"{len(cands)} inbox notes; first three folders:",
      [c.relative_to(INBOX).parts[0] for c in cands[:3]])
assert cands, "no inbox candidates found"
assert cands[0].relative_to(INBOX).parts[0] == "Phone", "Phone must sort first"

In [ ]:
note = parse_note(cands[0])
header, body = note_preview(note)
print("HEADER:", header)
print("BODY  :", body[:5])
assert isinstance(header, list) and isinstance(body, list)

In [ ]:
#| eval: false
# Round-trip check for the frontmatter writer, on a throwaway file outside the vault.
import tempfile
with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "t.md"
    p.write_text("---\ntitle: Test\ntags:\n  - a\n---\n\nBody line\n", encoding="utf-8")
    set_frontmatter(p, {"note-status": "🌳done", "hd": "d6"})
    out = p.read_text(encoding="utf-8")
    print(out)
    assert "title: Test" in out and "  - a" in out, "existing frontmatter must survive verbatim"
    assert "note-status: 🌳done" in out and "hd: d6" in out
    assert out.rstrip().endswith("Body line")
    set_frontmatter(p, {"hd": "d6.75"})
    assert "hd: d6.75" in p.read_text(encoding="utf-8")
    assert p.read_text(encoding="utf-8").count("hd:") == 1, "must replace, not duplicate"
    print("frontmatter round-trip OK")